# Utilities for validation

Stratified cross-validation

Stratified cross-validation ensures each fold keeps the same class proportions as the full dataset — useful when classes are imbalanced.

StratifiedKFold preserves class balance in each fold.

Note, after CV, refit on full X, y and then predict on test set



In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
import matplotlib.pyplot as plt

def perform_stratified_cross_val(model, X, y, score="accuracy", n_splits=5):
  cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

  # 1) CV accuracy
  acc = cross_val_score(model, X, y, cv=cv, scoring=score, n_jobs=-1)
  print(f"CV {score}: {acc.mean():.3f} ± {acc.std():.3f}")

  # 2) Out-of-fold predictions for diagnostics
  y_pred_oof = cross_val_predict(model, X, y, cv=cv, method="predict")

  cm = confusion_matrix(y, y_pred_oof)
  disp = ConfusionMatrixDisplay(cm, display_labels=["Died (0)", "Survived (1)"])
  disp.plot(cmap="Blues", values_format="d")
  plt.title("Confusion Matrix (OOF)"); plt.show()

  # (Optional) ROC-AUC with OOF probabilities
  y_proba_oof = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]
  print(f"CV ROC-AUC: {roc_auc_score(y, y_proba_oof):.3f}")